# Contruction de graphe et filtrage

## Import librairie et data

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import math
from textwrap import fill

PDB_PATH = "../data/raw/pdb_entry_results.csv"
SUM_PATH = "../data/raw/ppi3d_actin_summary.csv"

df = pd.read_csv(PDB_PATH, sep=";")
df_summary = pd.read_csv(SUM_PATH, sep=";")

## 1er filtre : garder uniquement les interactinos proteines-proteines

In [ ]:
print("Nombre de lignes dans df pdb :", len(df))
print("Nombre de lignes dans df_summary :", len(df_summary))

print(df["Interface type"].unique())
print(df_summary["Interface type"].unique())

# garder seulement protéine-protéine
df = df[df["Interface type"].isin(["hetero", "homo"])]
df_summary = df_summary[df_summary["Interface type"].isin(["hetero", "homo"])]
print(df["Interface type"].unique())
print(df_summary["Interface type"].unique())

print("Nombre de lignes dans df avec juste interface protéine-protéine:",
      len(df))
print("Nombre de lignes dans df_summary avec juste interface", 
      " protéine-protéine:", len(df_summary))

Nombre de lignes dans df pdb : 7262
Nombre de lignes dans df_summary : 5582
['hetero' 'homo' 'homo, inter-domain' 'hetero, inter-domain'
 'hetero, peptide' 'nucleic acid']
['hetero' 'homo']
['hetero' 'homo']
['hetero' 'homo']
Nombre de lignes dans df avec juste interface protéine-protéine: 6205
Nombre de lignes dans df_summary avec juste interface  protéine-protéine: 5582


## Création du graphe + 2eme filtrage (>= 5 actines connectées)

In [ ]:
graphs = {}
node_titles_dict = {}
node_labels_dict = {}
actin_nodes_dict = {}

valid_pdb = []
invalid_pdb = []

df_summary["Expect value"] = pd.to_numeric(df_summary["Expect value"], errors="coerce")
df["pdb_id"] = df["pdb_id"].astype(str).str.upper()

def normalize_title(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).strip().lower().split())

def display_label(node_id):
    suffix = str(node_id).split("_")[-1]
    return suffix[-1] if suffix else str(node_id)

target_proteins = set(
    df_summary.loc[df_summary["Expect value"] == 0.0, "Result protein"]
    .dropna()
    .map(normalize_title)
)

for pdb in df["pdb_id"].unique():
    sub = df[df["pdb_id"] == pdb]

    G = nx.Graph()
    node_titles = {}
    node_labels = {}

    for _, row in sub.iterrows():
        a = row["Interactor 1"]
        b = row["Interactor 2"]

        title_a = row["Interactor 1 title"]
        title_b = row["Interactor 2 title"]

        node_titles[a] = title_a
        node_titles[b] = title_b

        node_labels[a] = fill(title_a, width=20)
        node_labels[b] = fill(title_b, width=20)

    G.add_edge(a, b)

    graphs[pdb] = G
    node_titles_dict[pdb] = node_titles
    node_labels_dict[pdb] = node_labels

    actin_nodes = [
        node for node, title in node_titles.items()
        if normalize_title(title) in target_proteins
    ]
    actin_nodes_dict[pdb] = actin_nodes

    G_actin = G.subgraph(actin_nodes)

    keep = False
    if len(actin_nodes) > 0:
        keep = any(len(comp) >= 5 for comp in nx.connected_components(G_actin))

    if keep:
        valid_pdb.append(pdb)
    else:
        invalid_pdb.append(pdb)

print("PDB retenus :", len(valid_pdb))
print("PDB rejetés :", len(invalid_pdb))


PDB retenus : 0
PDB rejetés : 513


## Visualisation des graphes retenues

In [ ]:


pdb_ids = valid_pdb

n_cols = 3
n_rows = math.ceil(len(pdb_ids) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
axes = axes.flatten()

fig.suptitle(
    'PDB structures avec >= 5 actines',
    fontsize=16
)

for i, pdb in enumerate(pdb_ids):

    G = graphs[pdb]
    labels = node_labels_dict[pdb]
    actin_nodes = set(actin_nodes_dict[pdb])

    colors = [
        "red" if node in actin_nodes else "lightblue"
        for node in G.nodes()
    ]

    pos = nx.spring_layout(G, seed=42)

    nx.draw(
        G,
        pos,
        labels=labels,
        with_labels=True,
        node_color=colors,
        node_size=600,
        font_size=14,
        font_weight='bold',
        ax=axes[i]
    )

    axes[i].set_title(pdb)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout(rect=[0, 0, 1, 0.98])

plt.show()


ValueError: Number of rows must be a positive integer, not 0

<Figure size 1200x0 with 0 Axes>

## Créer les df filtrés

In [ ]:
# filtrer df pdb et summary
df_filtered = df[df["pdb_id"].isin(valid_pdb)]
print(f"df pdb: {len(df)} -> {len(df_filtered)}")

valid_pdb_lower = [p.lower() for p in valid_pdb]
df_summary_filtered = df_summary[df_summary["PDB ID"].isin(valid_pdb_lower)]
print(f"df summary: {len(df_summary)} -> {len(df_summary_filtered)}")

df pdb: 6205 -> 2393
df summary: 5582 -> 3497


In [ ]:
OUTPUT_SUMMARY = "../data/filtered/filtered_summary.csv"
OUTPUT_PDB_ENTRY = "../data/filtered/filtered_pdb_entry.csv"

df_filtered.to_csv(OUTPUT_PDB_ENTRY, index=False)
df_summary_filtered.to_csv(OUTPUT_SUMMARY, index=False)

In [ ]:
# Sauvegarder les graphes individuels + CSV protéines par PDB
import os

output_dir_pdb = "../visualisations/pdb_graphs"
os.makedirs(output_dir_pdb, exist_ok=True)

for pdb in valid_pdb:
    G = graphs[pdb]
    labels = node_labels_dict[pdb]
    actin_nodes = set(actin_nodes_dict[pdb])

    colors = ["red" if node in actin_nodes else "lightblue" for node in G.nodes()]

    fig, ax = plt.subplots(figsize=(6, 5))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(
        G, pos,
        labels=labels,
        with_labels=True,
        node_color=colors,
        node_size=600,
        font_size=14,
        font_weight="bold",
        font_color="black",
        ax=ax,
    )
    ax.set_title(pdb, fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{output_dir_pdb}/{pdb}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)

print(f"Graphes individuels sauvegardés pour {len(valid_pdb)} PDB → {output_dir_pdb}/")

# CSV : une ligne par chaîne unique, avec flag is_actin
rows = []
for pdb in valid_pdb:
    actin_nodes = set(actin_nodes_dict[pdb])
    for chain, title in node_titles_dict[pdb].items():
        rows.append({
            "pdb_id": pdb,
            "chain": chain,
            "protein": title,
            "is_actin": chain in actin_nodes,
        })

df_proteins_pdb = pd.DataFrame(rows)
df_proteins_pdb.to_csv("../data/filtered/proteins_per_pdb.csv", index=False)
print(f"proteins_per_pdb.csv sauvegardé ({len(df_proteins_pdb)} lignes)")

Graphes individuels sauvegardés pour 148 PDB → ../visualisations/pdb_graphs/
proteins_per_pdb.csv sauvegardé (1309 lignes)
